# Cross-Validation Analysis: BASELINE vs CVTEST_V2
## Calpella Daily - Effect of CV-based Hyperparameter Selection

**Research Question:** Does using cross-validation for hyperparameter selection improve test performance compared to single train/val split scoring?

**Runs:**
- `BASELINE`: Single train/val split scoring for hyperparam selection
- `CVTEST_V2`: CV-averaged scoring (3-fold) for hyperparam selection

In [1]:
import sys
sys.path.insert(0, "../..")

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from UCB_training.UCB_eval import load_test_metrics, pairwise_pct_change, threshold_filter

OUTPUT_BASE = Path("../../outputs/calpella")
DAILY_METRICS = OUTPUT_BASE / "daily"

BASELINE_DIR = DAILY_METRICS / "BASELINE_20250815T000000Z"
CV_DIR = DAILY_METRICS / "CVTEST_V2_20250815T000000Z"

print("Paths configured.")

Paths configured.


## 1. Load Test Metrics

In [2]:
baseline_metrics = load_test_metrics(BASELINE_DIR, "calpella")
cv_metrics = load_test_metrics(CV_DIR, "calpella")

print("BASELINE metrics loaded:", baseline_metrics.shape)
print("CVTEST_V2 metrics loaded:", cv_metrics.shape)

BASELINE metrics loaded: (14, 3)
CVTEST_V2 metrics loaded: (14, 3)


## 2. Full Comparison Table

In [3]:
MODELS = ["LSTM", "PILSTM"]

comparison = pairwise_pct_change(baseline_metrics, cv_metrics, models=MODELS)
comparison = comparison.rename(columns={"EXPERIMENTAL": "CVTEST_V2"})
comparison.round(3)

,Metric,Model,BASELINE,CVTEST_V2,pct_change
0,NSE,LSTM,0.802,0.804,0.268
1,NSE,PILSTM,0.841,0.848,0.841
2,MSE,LSTM,50422.880,49876.081,1.084
3,MSE,PILSTM,40461.058,38659.614,4.452
4,RMSE,LSTM,224.550,223.330,0.544
5,RMSE,PILSTM,201.149,196.620,2.251
6,KGE,LSTM,0.723,0.820,13.312
7,KGE,PILSTM,0.833,0.859,3.046
8,Alpha-NSE,LSTM,0.750,0.875,16.632
9,Alpha-NSE,PILSTM,0.876,0.890,1.615


## 3. Threshold Filtering

Filter to metrics with >5% change, always keeping NSE.

In [4]:
THRESHOLD = 5  # percent
ALWAYS_KEEP = ["NSE"]

comparison_with_pct = pairwise_pct_change(baseline_metrics, cv_metrics, models=MODELS)
filtered = threshold_filter(comparison_with_pct, threshold=THRESHOLD, always_keep=ALWAYS_KEEP)
filtered = filtered.rename(columns={"EXPERIMENTAL": "CVTEST_V2"})

print(f"Kept {len(filtered)} of {len(comparison_with_pct)} rows (threshold={THRESHOLD}%, always keep: {ALWAYS_KEEP})")
filtered.round(3)

Kept 12 of 28 rows (threshold=5%, always keep: ['NSE'])


,Metric,Model,BASELINE,CVTEST_V2
0,NSE,LSTM,0.802,0.804
1,NSE,PILSTM,0.841,0.848
6,KGE,LSTM,0.723,0.820
8,Alpha-NSE,LSTM,0.750,0.875
13,Beta-NSE,PILSTM,-0.045,-0.024
16,FHV,LSTM,-23.218,-10.334
18,FMS,LSTM,-29.641,-18.615
19,FMS,PILSTM,-12.444,-13.980
20,FLV,LSTM,-78.330,41.415
21,FLV,PILSTM,54.654,13.352


## 4. Key Findings

*[To be filled after running cells]*